# NephroLLM — Execution Plan

1. Environment setup
2. Load Llama 3.2 3B Instruct
3. Define & Verify the base-model Inference
4. Load and validate the nephrology benchmark
5. Run a fresh baseline evaluation
6. Analyze baseline failures
7. Prepare fine-tuning dataset
8. Fine-tune using QLoRA
9. Save the LoRA adapter
10. Run the exact same evaluation on the fine-tuned model
11. Evaluate on held-out questions
12. Compare base vs fine-tuned model
13. Document findings, limitations, and conclusions

# NephroLLM — Fine-Tuning Llama 3.2 3B for Nephrology Q&A

Educational fine-tuning experiment using QLoRA to adapt Llama 3.2 3B Instruct to nephrology-focused questions.

**Objective:** Compare the base model against the QLoRA fine-tuned model using the same evaluation benchmark and evaluation protocol.

# 1. Environment Setup - Install Required Libraries

This cell installs the libraries required for model loading, inference, dataset handling, quantization, and parameter-efficient fine-tuning.

We use:

- **Transformers** - load and run Llama 3.2 3B Instruct
- **Datasets** - prepare and manage training/evaluation data
- **Accelerate** - support efficient model execution on the available GPU
- **PEFT** - implement LoRA/QLoRA fine-tuning
- **BitsAndBytes** - enable 4-bit quantization
- **TRL** - provide utilities for supervised fine-tuning

The goal at this stage is only to prepare the environment. No model weights are modified or fine-tuned in this cell.

# 1.1 Install required libraries

In [8]:
# cell 2
!pip install -q -U transformers datasets accelerate peft bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 82.9 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 41.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.0/925.0 kB 21.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 61.7 MB/s eta 0:00:00:00:01


# 1.2 Verify environment and GPU
Environment and Hardware Verification

Before loading the model, we verify the versions of the main libraries and confirm that CUDA and the Kaggle GPU are available.

This is important because NephroLLM uses 4-bit quantization and will later use QLoRA for parameter-efficient fine-tuning. Both depend on compatible versions of the deep-learning and quantization libraries and access to a CUDA-enabled GPU.

We also record the available GPU memory so that the later fine-tuning configuration can be chosen with the available hardware in mind.

**Expected outcome:** CUDA should be available and a GPU should be detected.

In [9]:
# cell 4
import torch
import transformers
import datasets
import peft
import bitsandbytes

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

PyTorch: 2.10.0+cu128
Transformers: 5.17.0
Datasets: 5.0.1
PEFT: 0.21.0
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


# 1.3 Authenticate with Hugging Face

Llama 3.2 3B Instruct is accessed through Hugging Face and requires an authenticated account with access to the model.

This cell retrieves the Hugging Face access token from Kaggle Secrets rather than hard-coding credentials in the notebook.

We only verify that a token is available. The token itself is never printed.

**Why this matters:** keeping credentials in Kaggle Secrets prevents sensitive authentication information from being committed to the notebook or exposed in the GitHub repository.

In [10]:
# cell 6
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

hf_token = user_secrets.get_secret("HF_TOKEN")

print("HF token available:", hf_token is not None)
print("Token length:", len(hf_token) if hf_token else 0)

HF token available: True
Token length: 37


# 2 - Load Llama 3.2 3B Instruct

### Loading the Base Model with 4-bit Quantization

We now load the pretrained **Llama 3.2 3B Instruct** model that will serve as the starting point for the NephroLLM experiment.

The model is loaded using **4-bit NF4 quantization** with double quantization. This reduces the GPU memory required to load the 3B-parameter model and makes the later QLoRA experiment practical on the available Kaggle GPU.

At this stage, the model is still completely **untuned**. This is our base model and will be evaluated before any nephrology-specific fine-tuning.

**Configuration**

- **Base model:** `meta-llama/Llama-3.2-3B-Instruct`
- **Quantization:** 4-bit
- **Quantization type:** NF4
- **Compute dtype:** float16
- **Double quantization:** enabled
- **Device mapping:** automatic

**Why 4-bit quantization?**

QLoRA combines low-bit quantization of the base model with LoRA adapters that contain the trainable parameters. This allows us to fine-tune a large language model while using substantially less GPU memory than full-parameter fine-tuning.

**Expected outcome:** the tokenizer and quantized base model should load successfully on the available GPU.

In [11]:
# cell 8 
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from kaggle_secrets import UserSecretsClient

# Get Hugging Face token securely
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

model_id = "meta-llama/Llama-3.2-3B-Instruct"

# 4-bit quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    token=hf_token
)

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token
)

print("Model loaded successfully!")
print("Model device:", model.device)

Loading tokenizer...


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model loaded successfully!
Model device: cuda:0


# Experimental Design

### Base Model → Fine-Tuning → Re-evaluation

The goal of NephroLLM is to measure whether nephrology-specific fine-tuning improves the model's performance on multiple-choice nephrology questions.

The experiment follows a controlled evaluation design:

1. **Base model evaluation**  
   Run the untuned Llama 3.2 3B Instruct model on the **636-question primary evaluation benchmark**.

2. **Baseline error analysis**  
   Analyze incorrect answers, non-answers, response formatting, and performance across nephrology subjects.

3. **Training data preparation**  
   Prepare a **separate nephrology training corpus** that is not part of the 636-question primary benchmark or the 160-question held-out benchmark.

4. **QLoRA fine-tuning**  
   Fine-tune the base model using parameter-efficient QLoRA rather than updating all model parameters.

5. **Post-training evaluation**  
   Run the fine-tuned model on the **exact same 636 primary questions** using the same evaluation protocol.

6. **Held-out evaluation**  
   Evaluate the fine-tuned model separately on **160 held-out nephrology questions** that were not used for training.

7. **Comparison**  
   Compare base-model and fine-tuned performance using the same scoring methodology.

### Evaluation Split

| Split | Questions | Purpose |
|---|---:|---|
| Primary evaluation | 636 | Base vs fine-tuned comparison |
| Held-out evaluation | 160 | Generalization check |
| **Total evaluation data** | **796** | Evaluation only |

**Important:** the 636 primary questions and 160 held-out questions are evaluation data. They must not be used as QLoRA training examples.

This separation is important because using evaluation questions for training would introduce data leakage and make the before/after comparison unreliable.

**Expected outcome:** establish a reproducible experimental protocol where the only intended change between the baseline and post-training evaluation is the model's fine-tuned parameters.

# 3 - Define & Verify the base Model Inference 

# 3.1 Define standardized inference

Before running the benchmark, we define a reusable inference function that sends questions to the model using the Llama 3.2 Instruct chat format.

The function applies the model's chat template, generates a deterministic response, and returns only the newly generated text rather than the original prompt.

### Generation protocol

- **Sampling:** disabled (`do_sample=False`)
- **Maximum new tokens:** configurable per evaluation
- **Padding:** uses the tokenizer's EOS token
- **Inference mode:** gradients disabled with `torch.no_grad()`

Disabling sampling is important for evaluation because it makes the model's responses deterministic under the same configuration. This allows the base-model and fine-tuned evaluations to use the same generation protocol.

The function will be reused throughout the experiment rather than creating separate inference implementations for different stages.

**Expected outcome:** a single standardized inference function that can be used for smoke tests, benchmark evaluation, and post-training evaluation.

In [12]:
# cell 11
def ask_model(question, max_new_tokens=256):
    messages = [
        {
            "role": "system",
            "content": (
                "You are an educational assistant specializing in nephrology. "
                "Provide clear, factual, educational explanations. "
                "Do not provide diagnosis or personalized medical advice."
            )
        },
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip()

# 3.2 Run an inference smoke test

Before running the full benchmark, we perform a small smoke test using a simple nephrology question.

The purpose of this test is **not to measure model performance**. It verifies that the complete inference pipeline is working correctly:

- the tokenizer can format the prompt,
- the model can generate a response,
- the GPU inference path is functioning,
- the generated tokens can be decoded correctly, and
- the `ask_model()` function returns usable text.

This provides a quick sanity check before starting the longer 636-question baseline evaluation.

**Expected outcome:** the model should return a coherent educational response to the test question.

In [13]:
# cell 13
question = "What is the primary function of the glomerulus?"

answer = ask_model(question)

print("QUESTION:")
print(question)

print("\nMODEL ANSWER:")
print(answer)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


QUESTION:
What is the primary function of the glomerulus?

MODEL ANSWER:
The primary function of the glomerulus is to filter waste and excess fluids from the blood. It is a critical component of the nephron, the functional unit of the kidney, and plays a vital role in maintaining overall kidney function.

The glomerulus is a tuft of capillaries that receives blood from the afferent arteriole and sends it to the efferent arteriole. As the blood flows through the glomerulus, it is subjected to a pressure gradient that forces the fluid to pass through the glomerular basement membrane and into the Bowman's capsule. This process is known as filtration.

The glomerulus filters out waste products, excess ions, and excess water from the blood, while retaining essential nutrients, proteins, and other vital substances. The filtered waste products, including urea, creatinine, and other waste products, are then transported to the renal tubules for further processing and excretion.

In essence, the

# 4. Load and validate the Nephrology Benchmark

# 4.1 Experimental and data strategy


### Separating Training Data from Evaluation Data

A key requirement of this experiment is to keep the evaluation questions completely separate from the fine-tuning data.

The NephroLLM benchmark currently provides two evaluation splits:

- **Primary evaluation:** 636 questions
- **Held-out evaluation:** 160 questions

There is **no training split in this benchmark**. Therefore, these 796 questions will not be used as QLoRA training examples.

### Role of Each Dataset

| Dataset | Size | Role |
|---|---:|---|
| Primary evaluation | 636 | Base-model baseline and post-training comparison |
| Held-out evaluation | 160 | Generalization evaluation after fine-tuning |
| Separate training corpus | TBD | QLoRA fine-tuning |

The separate training corpus must contain nephrology educational material that is independent of these evaluation questions.

### Why this separation matters

If benchmark questions were included in the fine-tuning data, the model could memorize questions or answers rather than demonstrate genuine improvement in nephrology knowledge.

Keeping the evaluation data unseen during training allows us to make a cleaner before-versus-after comparison and provides a more meaningful test of generalization.

**Important:** at this stage, the evaluation benchmark is available, but the separate QLoRA training corpus still needs to be prepared.

**Expected outcome:** establish a clear data boundary before beginning fine-tuning and prevent evaluation-data leakage.

# 4.2 Load the primary evaluation benchmark

### Evaluating the Base Model on the Primary Benchmark

We now evaluate the untuned Llama 3.2 3B Instruct model on the **636-question primary nephrology benchmark**.

This establishes the quantitative baseline that will later be compared with the QLoRA fine-tuned model.

For every question, we preserve the model's raw response along with the parsed answer and ground-truth answer. This allows us to distinguish between:

- an incorrect answer,
- a response that could not be parsed,
- and a genuine non-answer or refusal.

### Evaluation protocol

The same protocol will be reused after fine-tuning:

- **Questions:** 636 primary evaluation questions
- **Model:** Llama 3.2 3B Instruct
- **Sampling:** deterministic (`do_sample=False`)
- **Maximum new tokens:** 96
- **Prompt format:** standardized multiple-choice prompt
- **Output:** raw model response + extracted answer + ground truth
- **Scoring:** exact option-level accuracy

The baseline results will serve as the reference point for measuring any change after QLoRA fine-tuning.

**Expected outcome:** a complete baseline prediction file containing one evaluation record for each of the 636 benchmark questions.

In [14]:
# cell 16
import os
import pandas as pd

# Find the finalized primary evaluation file
matches = []

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file == "nephrollm_primary_eval.csv":
            matches.append(os.path.join(root, file))

print("Found:", matches)

if len(matches) != 1:
    raise ValueError(
        f"Expected exactly one primary evaluation file, found: {matches}"
    )

eval_path = matches[0]
eval_df = pd.read_csv(eval_path)

print("Evaluation file:", eval_path)
print("Questions loaded:", len(eval_df))

display(
    eval_df[
        ["BenchmarkID", "Subject", "Question", "Choices", "Answer"]
    ].head()
)

Found: ['/kaggle/input/datasets/shantanuss/nejm-ai-nephrology-benchmark/nephrollm_primary_eval.csv']
Evaluation file: /kaggle/input/datasets/shantanuss/nejm-ai-nephrology-benchmark/nephrollm_primary_eval.csv
Questions loaded: 636


,BenchmarkID,Subject,Question,Choices,Answer
0,NEPHRO-EVAL-0001,Acute Kidney Injury & Critical Care Nephrology,Question 2824:\nThe transfusion of longer-term...,1. Increased risk of transfusion-related react...,B
1,NEPHRO-EVAL-0002,Acute Kidney Injury & Critical Care Nephrology,"Based on data from the NUTRIREA-2 study, enter...",A. Decreased incidence of systemic bacteremia\...,B
2,NEPHRO-EVAL-0003,Acute Kidney Injury & Critical Care Nephrology,Which ONE of the following is CORRECT about th...,1. She is at decreased risk for inpatient mort...,B
3,NEPHRO-EVAL-0004,Acute Kidney Injury & Critical Care Nephrology,Which ONE of the following is associated with ...,A. A decreased risk of surgical site infection...,B
4,NEPHRO-EVAL-0005,Acute Kidney Injury & Critical Care Nephrology,Which ONE of the following is the MOST importa...,A. Administration of a bolus of 30 ml/kg 0.9% ...,D


# 5. Run a Fresh Baseline Evaluation

# 5.1 Lock the evaluation protocol
Locking the Evaluation Configuration

Before running the full baseline, we explicitly define the evaluation configuration.

This is important because the same evaluation protocol must be used later when testing the QLoRA fine-tuned model. Keeping the protocol fixed makes the before-versus-after comparison reproducible.

For each question, we will preserve:

1. The original benchmark question
2. The available answer choices
3. The model's raw response
4. The answer extracted by the evaluator
5. The ground-truth answer
6. Whether the extracted answer is correct

### Fixed evaluation settings

| Setting | Configuration |
|---|---|
| Evaluation split | Primary |
| Number of questions | 636 |
| Generation | Deterministic |
| `do_sample` | `False` |
| `max_new_tokens` | `96` |
| Scoring | Exact option match |
| Raw responses | Saved |
| Parsed answers | Saved |
| Ground truth | Saved |

The evaluator is intentionally conservative: if the model does not provide a recognizable answer choice, the response is treated as **unparsed/non-answer** rather than guessing what the model intended.

**Expected outcome:** the evaluation configuration is clearly documented before the baseline results are generated.

# 5.2 Define the MCQ answer parser

Multiple-Choice Answer Evaluator

The benchmark contains multiple-choice questions with four or five answer options. The model may express its answer in different formats, such as:

- `ANSWER: A`
- `ANSWER: 2`
- `C`
- `Option D`
- `Choice: B`

The evaluator normalizes these responses into a standard option label (`A`–`E`) before comparing them with the benchmark's ground-truth answer.

### Conservative parsing

The parser prioritizes explicit answer statements and recognizable option formats.

It does **not** attempt to infer an answer from general prose when the model does not clearly select an option.

For example, if the model produces a refusal, uncertainty statement, or explanation without a clearly identifiable answer choice, the response is recorded as **unparsed/non-answer**.

This conservative approach is intentional. Guessing the model's intended answer could artificially inflate the measured accuracy.

### Scoring

For each question:

```text
Predicted option == Ground-truth option
            ↓
        Correct / Incorrect

In [15]:
import re

def extract_answer(response):
    """
    Extract a single answer choice.

    Supports alphabetical and numerical answer formats.
    Numerical choices are mapped by position:
    1=A, 2=B, 3=C, 4=D, 5=E
    """

    if not response:
        return None

    text = str(response).strip().upper()

    # 1. Explicit ANSWER / CORRECT ANSWER formats
    patterns = [
        r"\bANSWER\s*:\s*([A-E])\b",
        r"\bCORRECT ANSWER\s*(?:IS)?\s*:?\s*([A-E])\b",
        r"\bTHE ANSWER\s*(?:IS)?\s*:?\s*([A-E])\b",
        r"\bMY ANSWER\s*(?:IS)?\s*:?\s*([A-E])\b",
    ]

    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            return match.group(1)

    # 2. Explicit numeric ANSWER format
    numeric_patterns = [
        r"\bANSWER\s*:\s*([1-5])\b",
        r"\bCORRECT ANSWER\s*(?:IS)?\s*:?\s*([1-5])\b",
        r"\bTHE ANSWER\s*(?:IS)?\s*:?\s*([1-5])\b",
        r"\bMY ANSWER\s*(?:IS)?\s*:?\s*([1-5])\b",
    ]

    for pattern in numeric_patterns:
        match = re.search(pattern, text)
        if match:
            return "ABCDE"[int(match.group(1)) - 1]

    # 3. Standalone answer formats: A, A., A)
    lines = [x.strip() for x in text.splitlines() if x.strip()]

    for line in lines[:8]:
        match = re.fullmatch(r"([A-E])[\.\)]?", line)
        if match:
            return match.group(1)

    # 4. Standalone numeric answer: 1, 1., 1)
    for line in lines[:8]:
        match = re.fullmatch(r"([1-5])[\.\)]?", line)
        if match:
            return "ABCDE"[int(match.group(1)) - 1]

    # 5. OPTION / CHOICE format
    for line in lines[:8]:
        match = re.fullmatch(
            r"(?:OPTION|CHOICE)\s*[:\-]?\s*([A-E])[\.\)]?",
            line
        )
        if match:
            return match.group(1)

        match = re.fullmatch(
            r"(?:OPTION|CHOICE)\s*[:\-]?\s*([1-5])[\.\)]?",
            line
        )
        if match:
            return "ABCDE"[int(match.group(1)) - 1]

    return None

# 5.3 Define the standardized MCQ prompt

We now define the prompt used to evaluate each benchmark question.

The model is explicitly instructed to select **one answer from the provided choices** and return it in a standardized format:

```text
ANSWER: <option letter>

In [16]:
# cell 19
def ask_mcq_fast(question, choices, max_new_tokens=32):
    """
    Ask the base model to answer one MCQ.

    The benchmark may contain 4 or 5 choices and may use
    alphabetical or numerical labels.
    """

    choice_lines = str(choices).strip()

    prompt = f"""Answer this multiple-choice nephrology question.

Question:
{question}

Choices:
{choice_lines}

Select the single best answer from the choices provided.

Return ONLY:
ANSWER: <option letter>
"""

    return ask_model(
        prompt,
        max_new_tokens=max_new_tokens
    )

# 5.4 Run a small evaluation sanity check


In [17]:
# cell 27 
test_df = eval_df.head(5).copy()

test_results = []

for _, row in test_df.iterrows():

    response = ask_mcq_fast(
        row["Question"],
        row["Choices"]
    )

    predicted = extract_answer(response)
    expected = str(row["Answer"]).strip().upper()

    test_results.append({
        "BenchmarkID": row["BenchmarkID"],
        "Expected": expected,
        "Predicted": predicted,
        "Correct": predicted == expected,
        "ModelResponse": response
    })

test_results_df = pd.DataFrame(test_results)

display(test_results_df)

,BenchmarkID,Expected,Predicted,Correct,ModelResponse
0,NEPHRO-EVAL-0001,B,D,False,D
1,NEPHRO-EVAL-0002,B,A,False,A
2,NEPHRO-EVAL-0003,B,A,False,A.
3,NEPHRO-EVAL-0004,B,B,True,B
4,NEPHRO-EVAL-0005,D,D,True,D


# 5.5 Run the full 636-question baseline

### Running the 636-Question Benchmark

The evaluation protocol is now locked, so we can run the complete primary benchmark against the untuned base model.

The benchmark contains **636 questions**. Because generation can be time-consuming on the available GPU, the evaluation is executed in batches and each batch is saved separately.

For every question, we retain the model's raw response and the information required for later scoring and error analysis.

### Why save batches?

Saving intermediate results provides two benefits:

1. **Fault tolerance:** if a long evaluation run is interrupted, completed batches do not need to be regenerated.
2. **Reproducibility:** each batch produces a persistent prediction file that can be independently inspected and aggregated.

After all batches are complete, the individual files are combined into one final baseline result set containing all 636 questions.

**Important:** this run is the base-model baseline. No QLoRA training has been performed yet.

**Expected outcome:** 636 baseline predictions are generated and stored for subsequent scoring and analysis.

In [18]:
from tqdm.auto import tqdm
import os
import pandas as pd

BATCH_SIZE = 50

BASELINE_PREFIX = "/kaggle/working/nephrollm_baseline_final_batch_"

for start in range(0, len(eval_df), BATCH_SIZE):

    end = min(start + BATCH_SIZE, len(eval_df))
    batch = eval_df.iloc[start:end].copy()

    batch_number = start // BATCH_SIZE + 1

    print(
        f"\nRunning Batch {batch_number}: "
        f"questions {start + 1} to {end}"
    )

    batch_results = []

    for _, row in tqdm(
        batch.iterrows(),
        total=len(batch),
        desc=f"Batch {batch_number}"
    ):

        response = ask_mcq_fast(
            row["Question"],
            row["Choices"]
        )

        predicted = extract_answer(response)
        expected = str(row["Answer"]).strip().upper()

        batch_results.append({
            "BenchmarkID": row["BenchmarkID"],
            "Subject": row["Subject"],
            "Question": row["Question"],
            "Choices": row["Choices"],
            "Expected": expected,
            "Predicted": predicted,
            "Correct": predicted == expected,
            "ModelResponse": response
        })

    batch_df = pd.DataFrame(batch_results)

    batch_path = (
        f"{BASELINE_PREFIX}"
        f"{start + 1:03d}_{end:03d}.csv"
    )

    batch_df.to_csv(batch_path, index=False)

    print(f"Saved: {batch_path}")
    print(
        f"Batch accuracy: "
        f"{batch_df['Correct'].mean():.2%}"
    )
    print(
        f"Unparsed: "
        f"{batch_df['Predicted'].isna().sum()}"
    )


Running Batch 1: questions 1 to 50


Batch 1:   0%|          | 0/50 [00:00<?, ?it/s]

Saved: /kaggle/working/nephrollm_baseline_final_batch_001_050.csv
Batch accuracy: 56.00%
Unparsed: 1

Running Batch 2: questions 51 to 100


Batch 2:   0%|          | 0/50 [00:00<?, ?it/s]

Saved: /kaggle/working/nephrollm_baseline_final_batch_051_100.csv
Batch accuracy: 48.00%
Unparsed: 1

Running Batch 3: questions 101 to 150


...

Saved: /kaggle/working/nephrollm_baseline_final_batch_501_550.csv
Batch accuracy: 26.00%
Unparsed: 5

Running Batch 12: questions 551 to 600


Batch 12:   0%|          | 0/50 [00:00<?, ?it/s]

Saved: /kaggle/working/nephrollm_baseline_final_batch_551_600.csv
Batch accuracy: 44.00%
Unparsed: 2

Running Batch 13: questions 601 to 636


Batch 13:   0%|          | 0/36 [00:00<?, ?it/s]

Saved: /kaggle/working/nephrollm_baseline_final_batch_601_636.csv
Batch accuracy: 33.33%
Unparsed: 1


# 5.6 Aggregate baseline predictions

The 636-question baseline was executed in multiple batches. We now combine those batch outputs into a single evaluation dataset.

Before calculating the final score, we verify that:

- all expected batch files are present,
- the combined result contains 636 rows,
- each benchmark question appears only once, and
- the final result can be saved as a single reproducible CSV.

This step converts the raw batch predictions into the canonical baseline result file that will be used for all subsequent analysis.

**Expected outcome:** one consolidated baseline file containing exactly **636 unique benchmark questions**.

In [19]:
import glob
import pandas as pd

baseline_files = sorted(
    glob.glob(
        "/kaggle/working/nephrollm_baseline_final_batch_*.csv"
    )
)

print("Fresh baseline batch files:", len(baseline_files))

if len(baseline_files) != 13:
    raise ValueError(
        f"Expected 13 fresh baseline files, found {len(baseline_files)}"
    )

baseline_df = pd.concat(
    [pd.read_csv(f) for f in baseline_files],
    ignore_index=True
)

# Safety checks
print("Rows before deduplication:", len(baseline_df))
print(
    "Unique BenchmarkIDs:",
    baseline_df["BenchmarkID"].nunique()
)

if len(baseline_df) != 636:
    raise ValueError(
        f"Expected 636 rows, found {len(baseline_df)}"
    )

if baseline_df["BenchmarkID"].nunique() != 636:
    raise ValueError("Duplicate or missing BenchmarkIDs detected.")

# Save one consolidated fresh baseline file
baseline_path = (
    "/kaggle/working/"
    "nephrollm_baseline_final_results.csv"
)

baseline_df.to_csv(
    baseline_path,
    index=False
)

print("\nFresh baseline saved:")
print(baseline_path)

print("\nColumns:")
print(list(baseline_df.columns))

Fresh baseline batch files: 13
Rows before deduplication: 636
Unique BenchmarkIDs: 636

Fresh baseline saved:
/kaggle/working/nephrollm_baseline_final_results.csv

Columns:
['BenchmarkID', 'Subject', 'Question', 'Choices', 'Expected', 'Predicted', 'Correct', 'ModelResponse']


# 6. Analyze Baseline Failures
### Measuring Base-Model Performance

With all 636 predictions consolidated, we now calculate the baseline performance of the untuned Llama 3.2 3B Instruct model.

We report two complementary metrics:

- **Accuracy (all questions):** correct answers divided by all 636 benchmark questions.
- **Accuracy (answered-only):** correct answers divided by questions for which the evaluator successfully extracted a valid answer choice.

We also report the number of **unparsed/non-answer responses** separately. This is important because a model can fail either by selecting the wrong option or by failing to provide a usable answer.

### Fresh baseline result

| Metric | Result |
|---|---:|
| Questions evaluated | ** ** |
| Correct | ** ** |
| Incorrect | ** ** |
| Unparsed / non-answer | ** ** |
| Accuracy — all questions | ** ** |
| Accuracy — answered only | ** ** |

The **41.51%** figure is the primary baseline metric because it treats a non-answer as a failure on the benchmark item rather than excluding it from evaluation.

The answered-only figure is reported as a secondary diagnostic metric to show performance among responses where the evaluator could identify a selected option.

These baseline results will become the reference point for the post-QLoRA evaluation.

**Important:** these are the results from the fresh 636-question baseline run and the finalized conservative answer parser. No fine-tuning has been performed yet.

# 6.1 Calculate the initial baseline results


In [20]:
print("NEPHROLLM — FRESH BASELINE")
print("==========================")

total_questions = len(baseline_df)
correct = baseline_df["Correct"].sum()
unparsed = baseline_df["Predicted"].isna().sum()
incorrect = total_questions - correct - unparsed

accuracy_all = correct / total_questions
accuracy_answered = (
    correct / (total_questions - unparsed)
    if total_questions - unparsed > 0
    else 0
)

print(f"Total questions:      {total_questions}")
print(f"Correct:              {correct}")
print(f"Incorrect:            {incorrect}")
print(f"Unparsed:             {unparsed}")
print(f"Accuracy (all):       {accuracy_all:.2%}")
print(f"Accuracy (answered):  {accuracy_answered:.2%}")

NEPHROLLM — FRESH BASELINE
Total questions:      636
Correct:              264
Incorrect:            328
Unparsed:             44
Accuracy (all):       41.51%
Accuracy (answered):  44.59%


# 6.2 Inspect unparsed responses

After applying the finalized answer parser, we inspect the responses that still cannot be mapped reliably to an answer choice.

The purpose of this step is to validate that the remaining parsing failures are genuine **non-answers or ambiguous responses**, rather than valid answer formats that the parser has missed.

We display a sample of the unparsed responses together with their expected answers and available choices. This allows us to manually inspect the model's actual output before treating these responses as failures in the final baseline score.

The unparsed responses are **not reinterpreted or guessed** during this inspection. If the model does not clearly select an answer option, the response remains unparsed.

**Expected outcome:** confirm that the remaining unparsed responses represent genuine non-answers or ambiguous outputs and that the conservative parser should remain unchanged.

In [21]:
unparsed_df = baseline_df[
    baseline_df["Predicted"].isna()
].copy()

print("Unparsed responses:", len(unparsed_df))

display(
    unparsed_df[
        [
            "BenchmarkID",
            "Expected",
            "Choices",
            "ModelResponse"
        ]
    ].head(20)
)

Unparsed responses: 44


,BenchmarkID,Expected,Choices,ModelResponse
24,NEPHRO-EVAL-0025,C,A. Intermittent hemodiaﬁltration\nB. Intermitt...,I'm happy to help you with your nephrology que...
64,NEPHRO-EVAL-0065,A,A. Regional citrate anticoagulation (RC...,I'm happy to help you with your nephrology que...
144,NEPHRO-EVAL-0145,D,A. Denosumab\nB. Increase calcium acetate\nC. ...,I'm happy to help you with your nephrology que...
155,NEPHRO-EVAL-0156,A,A. Cinacalcet therapy\nB. Calcitonin\nC. Estro...,I'm happy to help you with your nephrology que...
176,NEPHRO-EVAL-0177,C,1. Refer for cardiac catheterization\n2. Chest...,I'm happy to help you with your nephrology que...
180,NEPHRO-EVAL-0181,C,1. Ankylosing spondylitis\n2. Adynamic bone di...,I'm happy to help you with your nephrology que...
182,NEPHRO-EVAL-0183,D,1. Hypercalciuria\n2. Renal tubular acidosis\n...,I'm happy to help you with your nephrology que...
188,NEPHRO-EVAL-0189,C,1. A loss of function mutation in the calcium-...,I'm happy to help you with your nephrology que...
191,NEPHRO-EVAL-0192,A,1. Change calcium acetate to sevelamer carbona...,I can't provide a specific answer without know...
227,NEPHRO-EVAL-0228,D,A. Discontinue pravastatin now\nB. Discontinue...,I'm happy to help you with your nephrology que...


# Re-scoring the Complete Baseline

We now apply the finalized answer parser to all stored model responses and recompute the baseline metrics.

The scoring logic separates three outcomes:

- **Correct:** the parsed model answer matches the benchmark answer.
- **Incorrect:** the model selected an identifiable option, but it does not match the benchmark answer.
- **Unparsed:** the response does not contain a reliably identifiable answer choice.

Two accuracy measures are calculated:

- **Accuracy (all):** correct answers divided by all 636 questions.
- **Accuracy (answered):** correct answers divided by questions with a successfully parsed answer.

The re-scored results are saved to:

`nephrollm_baseline_final_results.csv`

This is a deterministic post-processing step. **No model inference is performed here**, so the underlying model responses remain exactly the same as those generated during the original baseline run.

**Expected outcome:** the finalized baseline metrics are produced using the locked conservative parser and the complete set of 636 stored responses.

In [22]:
baseline_df["Predicted"] = (
    baseline_df["ModelResponse"]
    .apply(extract_answer)
)

baseline_df["Correct"] = (
    baseline_df["Predicted"]
    == baseline_df["Expected"]
)

total_questions = len(baseline_df)
correct = baseline_df["Correct"].sum()
unparsed = baseline_df["Predicted"].isna().sum()
incorrect = total_questions - correct - unparsed

print("NEPHROLLM — FRESH BASELINE (REPARSED)")
print("=====================================")
print(f"Total questions:      {total_questions}")
print(f"Correct:              {correct}")
print(f"Incorrect:            {incorrect}")
print(f"Unparsed:             {unparsed}")
print(f"Accuracy (all):       {correct / total_questions:.2%}")
print(
    f"Accuracy (answered):  "
    f"{correct / (total_questions - unparsed):.2%}"
)

baseline_df.to_csv(
    "/kaggle/working/nephrollm_baseline_final_results.csv",
    index=False
)

NEPHROLLM — FRESH BASELINE (REPARSED)
Total questions:      636
Correct:              264
Incorrect:            328
Unparsed:             44
Accuracy (all):       41.51%
Accuracy (answered):  44.59%


# Final Inspection of Unparsed Responses

After re-scoring the complete baseline with the finalized parser, we perform one final inspection of the responses that remain unparsed.

This confirms that the parser is not systematically missing a valid answer format that should be counted.

For each remaining unparsed response, we retain:

- `BenchmarkID`
- expected answer
- available choices
- raw model response

The responses are kept as unparsed when the model does not provide a sufficiently clear option selection. We do not manually assign an answer during scoring.

This preserves a conservative and reproducible evaluation protocol.

**Expected outcome:** the remaining unparsed responses are confirmed as genuine non-answers or ambiguous outputs, supporting the use of the finalized baseline score.

In [23]:
remaining_unparsed = baseline_df[
    baseline_df["Predicted"].isna()
].copy()

print("Remaining unparsed:", len(remaining_unparsed))

display(
    remaining_unparsed[
        [
            "BenchmarkID",
            "Expected",
            "Choices",
            "ModelResponse"
        ]
    ]
)

Remaining unparsed: 44


,BenchmarkID,Expected,Choices,ModelResponse
24,NEPHRO-EVAL-0025,C,A. Intermittent hemodiaﬁltration\nB. Intermitt...,I'm happy to help you with your nephrology que...
64,NEPHRO-EVAL-0065,A,A. Regional citrate anticoagulation (RC...,I'm happy to help you with your nephrology que...
144,NEPHRO-EVAL-0145,D,A. Denosumab\nB. Increase calcium acetate\nC. ...,I'm happy to help you with your nephrology que...
155,NEPHRO-EVAL-0156,A,A. Cinacalcet therapy\nB. Calcitonin\nC. Estro...,I'm happy to help you with your nephrology que...
176,NEPHRO-EVAL-0177,C,1. Refer for cardiac catheterization\n2. Chest...,I'm happy to help you with your nephrology que...
180,NEPHRO-EVAL-0181,C,1. Ankylosing spondylitis\n2. Adynamic bone di...,I'm happy to help you with your nephrology que...
182,NEPHRO-EVAL-0183,D,1. Hypercalciuria\n2. Renal tubular acidosis\n...,I'm happy to help you with your nephrology que...
188,NEPHRO-EVAL-0189,C,1. A loss of function mutation in the calcium-...,I'm happy to help you with your nephrology que...
191,NEPHRO-EVAL-0192,A,1. Change calcium acetate to sevelamer carbona...,I can't provide a specific answer without know...
227,NEPHRO-EVAL-0228,D,A. Discontinue pravastatin now\nB. Discontinue...,I'm happy to help you with your nephrology que...


# 6.5 Analyze performance by subject

### Performance Across Nephrology Subjects

Overall accuracy provides a useful headline metric, but it does not show whether the model performs consistently across different areas of nephrology.

We therefore break the 636-question baseline down by the benchmark's **Subject** field.

For each subject, we examine:

- number of questions,
- number of correct answers,
- number of unparsed responses, and
- accuracy.

This helps identify areas where the base model performs relatively differently and provides a more informative starting point for evaluating the effect of nephrology-specific fine-tuning.

### Important interpretation note

Subject-level results should be interpreted alongside the number of questions in each category. Smaller subject groups can produce larger percentage changes from only a few questions.

The subject labels are retained exactly as provided by the benchmark rather than manually merging or renaming categories.

**Expected outcome:** a subject-level baseline performance table showing where the base model succeeds and where errors are concentrated.

In [24]:
subject_accuracy = (
    baseline_df
    .groupby("Subject")
    .agg(
        Questions=("BenchmarkID", "count"),
        Correct=("Correct", "sum"),
        Unparsed=("Predicted", lambda x: x.isna().sum())
    )
    .reset_index()
)

subject_accuracy["Incorrect"] = (
    subject_accuracy["Questions"]
    - subject_accuracy["Correct"]
    - subject_accuracy["Unparsed"]
)

subject_accuracy["Accuracy"] = (
    subject_accuracy["Correct"]
    / subject_accuracy["Questions"]
    * 100
)

subject_accuracy = subject_accuracy.sort_values(
    "Accuracy",
    ascending=True
)

display(subject_accuracy)


,Subject,Questions,Correct,Unparsed,Incorrect,Accuracy
15,Primary & Secondary Glomerular Diseases,23,6,7,10,26.086957
17,Primary and Secondary Glomerular Diseases,48,13,5,30,27.083333
4,Disorders of Divalent Ions Renal Bone Disease ...,22,6,2,14,27.272727
14,Pregnancy and Kidney Disease,24,7,3,14,29.166667
16,Primary Glomerular Disease,24,7,2,15,29.166667
7,Electrolyte Acid-Base Disorders,24,7,5,12,29.166667
5,"Disorders of Divalent Ions, Renal Bone Disease...",46,14,5,27,30.434783
8,Electrolytes and Acid-Base Disorders,22,7,2,13,31.818182
12,Infection Control & Prevention in Outpatient H...,24,9,2,13,37.500000
19,Transplantation,48,19,1,28,39.583333


# 6.6 Analyze incorrect answer selections


In [25]:
# Baseline error analysis — parseable incorrect answers only

baseline_errors = baseline_df[
    (baseline_df["Predicted"].notna()) &
    (baseline_df["Correct"] == False)
].copy()

print("BASELINE ERROR ANALYSIS")
print("=======================")
print(f"Total benchmark questions: {len(baseline_df)}")
print(f"Parseable incorrect answers: {len(baseline_errors)}")
print()

print("Most common predicted answers:")
display(
    baseline_errors["Predicted"]
    .value_counts()
    .rename_axis("Predicted")
    .reset_index(name="Count")
)

print("\nExpected answer distribution:")
display(
    baseline_errors["Expected"]
    .value_counts()
    .rename_axis("Expected")
    .reset_index(name="Count")
)

print("\nSample baseline errors:")
display(
    baseline_errors[
        [
            "BenchmarkID",
            "Subject",
            "Question",
            "Expected",
            "Predicted",
            "ModelResponse"
        ]
    ].head(20)
)

BASELINE ERROR ANALYSIS
Total benchmark questions: 636
Parseable incorrect answers: 328

Most common predicted answers:


,Predicted,Count
0,B,123
1,A,84
2,C,55
3,D,53
4,E,13



Expected answer distribution:


,Expected,Count
0,D,95
1,C,89
2,B,73
3,A,57
4,E,14



Sample baseline errors:


,BenchmarkID,Subject,Question,Expected,Predicted,ModelResponse
0,NEPHRO-EVAL-0001,Acute Kidney Injury & Critical Care Nephrology,Question 2824:\nThe transfusion of longer-term...,B,D,D
1,NEPHRO-EVAL-0002,Acute Kidney Injury & Critical Care Nephrology,"Based on data from the NUTRIREA-2 study, enter...",B,A,A
2,NEPHRO-EVAL-0003,Acute Kidney Injury & Critical Care Nephrology,Which ONE of the following is CORRECT about th...,B,A,A.
5,NEPHRO-EVAL-0006,Acute Kidney Injury & Critical Care Nephrology,Which ONE of the following individuals is MOST...,B,C,ANSWER: C
6,NEPHRO-EVAL-0007,Acute Kidney Injury & Critical Care Nephrology,"Based on the existing literature, implementing...",B,D,ANSWER: D
7,NEPHRO-EVAL-0008,Acute Kidney Injury & Critical Care Nephrology,Which ONE of the following outcomes has been d...,C,B,B
8,NEPHRO-EVAL-0009,Acute Kidney Injury & Critical Care Nephrology,Question 2833:\nImplementing a Kidney Disease ...,B,D,ANSWER: D
16,NEPHRO-EVAL-0017,Acute Kidney Injury & Critical Care Nephrology,Which ONE of the following should you tell the...,B,A,ANSWER: A
27,NEPHRO-EVAL-0028,Acute Kidney Injury & Critical Care Nephrology,Which ONE of the following is MOST likely to r...,B,C,ANSWER: 3
29,NEPHRO-EVAL-0030,Acute Kidney Injury & Critical Care Nephrology,Which ONE of the following interventions would...,D,C,ANSWER: 3


# 6.7 Analyze response formats


In [26]:
# Analyze baseline response formats

def classify_response_format(response):
    if response is None or str(response).strip() == "":
        return "empty"

    text = str(response).strip().upper()

    if re.search(r"\bANSWER\s*:\s*[A-E]\b", text):
        return "ANSWER: letter"

    if re.search(r"\bANSWER\s*:\s*[1-5]\b", text):
        return "ANSWER: number"

    if re.fullmatch(r"[A-E][\.\)]?", text):
        return "standalone letter"

    if re.fullmatch(r"[1-5][\.\)]?", text):
        return "standalone number"

    if text.startswith("I CANNOT") or text.startswith("I CAN'T"):
        return "refusal"

    if "MEDICAL ADVICE" in text:
        return "medical-advice refusal"

    return "other"


baseline_df["ResponseFormat"] = (
    baseline_df["ModelResponse"]
    .apply(classify_response_format)
)

format_summary = (
    baseline_df
    .groupby("ResponseFormat")
    .agg(
        Questions=("BenchmarkID", "count"),
        Correct=("Correct", "sum")
    )
    .reset_index()
)

format_summary["Accuracy"] = (
    format_summary["Correct"]
    / format_summary["Questions"]
    * 100
)

format_summary = format_summary.sort_values(
    "Questions",
    ascending=False
)

display(format_summary)

,ResponseFormat,Questions,Correct,Accuracy
0,ANSWER: letter,233,94,40.343348
1,ANSWER: number,176,81,46.022727
4,standalone letter,172,85,49.418605
2,other,46,3,6.521739
3,refusal,8,0,0.000000
5,standalone number,1,1,100.000000


# 6.8 Audit generated baseline artifacts

In [27]:
import os
import glob

print("Kaggle input datasets:")
print("======================")

for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files[:20]:
        print(f"{indent}  {file}")

print("\nWorking directory CSV files:")
print("============================")

for path in sorted(glob.glob("/kaggle/working/*.csv")):
    print(path)

Kaggle input datasets:
input/
  datasets/
    shantanuss/
      nejm-ai-nephrology-benchmark/
        nephrollm_evaluation_manifest.csv
        nephrollm_primary_eval.csv
        nephrollm_heldout_eval.csv

Working directory CSV files:
/kaggle/working/nephrollm_baseline_final_batch_001_050.csv
/kaggle/working/nephrollm_baseline_final_batch_051_100.csv
/kaggle/working/nephrollm_baseline_final_batch_101_150.csv
/kaggle/working/nephrollm_baseline_final_batch_151_200.csv
/kaggle/working/nephrollm_baseline_final_batch_201_250.csv
/kaggle/working/nephrollm_baseline_final_batch_251_300.csv
/kaggle/working/nephrollm_baseline_final_batch_301_350.csv
/kaggle/working/nephrollm_baseline_final_batch_351_400.csv
/kaggle/working/nephrollm_baseline_final_batch_401_450.csv
/kaggle/working/nephrollm_baseline_final_batch_451_500.csv
/kaggle/working/nephrollm_baseline_final_batch_501_550.csv
/kaggle/working/nephrollm_baseline_final_batch_551_600.csv
/kaggle/working/nephrollm_baseline_final_batch_601_636.c

# 4.3 Load the benchmark manifest and held-out dataset

### Loading the Evaluation Manifest and Held-Out Split

We now load the benchmark manifest and the separate held-out evaluation dataset.

The **manifest** provides the official dataset structure and split assignments, while the **held-out dataset** contains the 160 questions reserved for the final generalization evaluation.

At this stage, we are only inspecting the data. No held-out questions are used for model training.

We verify:

- dataset dimensions,
- available columns,
- sample records, and
- the structure of the held-out evaluation data.

### Why this matters

The held-out split must remain separate from the QLoRA training corpus. Keeping it untouched allows us to evaluate whether the fine-tuned model can generalize to nephrology questions that were not used during training.

The held-out questions will be evaluated **after fine-tuning**, using the same answer-extraction and scoring methodology used for the primary evaluation.

**Expected outcome:** the manifest and held-out evaluation dataset load successfully, with the held-out split containing **160 questions**.

In [28]:
import pandas as pd

BASE = "/kaggle/input/datasets/shantanuss/nejm-ai-nephrology-benchmark"

manifest_path = f"{BASE}/nephrollm_evaluation_manifest.csv"
heldout_path = f"{BASE}/nephrollm_heldout_eval.csv"

manifest_df = pd.read_csv(manifest_path)
heldout_df = pd.read_csv(heldout_path)

print("MANIFEST")
print("========")
print("Shape:", manifest_df.shape)
print("Columns:", list(manifest_df.columns))
display(manifest_df.head(20))

print("\nHELD-OUT EVALUATION")
print("===================")
print("Shape:", heldout_df.shape)
print("Columns:", list(heldout_df.columns))
display(heldout_df.head(10))

MANIFEST
Shape: (796, 10)
Columns: ['BenchmarkID', 'ID', 'Subject', 'Context', 'Question', 'Choices', 'Answer', 'Solution', 'explanation', 'split']


,BenchmarkID,ID,Subject,Context,Question,Choices,Answer,Solution,explanation,split
0,NEPHRO-EVAL-0001,2824,Acute Kidney Injury & Critical Care Nephrology,NaN,Question 2824:\nThe transfusion of longer-term...,1. Increased risk of transfusion-related react...,B,temp nan,Answer B: No difference in mortality at 90 days,primary_eval
1,NEPHRO-EVAL-0002,2825,Acute Kidney Injury & Critical Care Nephrology,A 79-year-old man with stage G4 CKD and recent...,"Based on data from the NUTRIREA-2 study, enter...",A. Decreased incidence of systemic bacteremia\...,B,temp nan,Answer B: Increased incidence of gastrointest...,primary_eval
2,NEPHRO-EVAL-0003,2826,Acute Kidney Injury & Critical Care Nephrology,A 67-year-old woman with a history of stage G2...,Which ONE of the following is CORRECT about th...,1. She is at decreased risk for inpatient mort...,B,temp nan,Answer B: Her duration of RRT will be shorter,primary_eval
3,NEPHRO-EVAL-0004,2827,Acute Kidney Injury & Critical Care Nephrology,You are caring for a 71-year-old man with stag...,Which ONE of the following is associated with ...,A. A decreased risk of surgical site infection...,B,temp nan,Answer B: An increased risk of AKI requiring RRT,primary_eval
4,NEPHRO-EVAL-0005,2829,Acute Kidney Injury & Critical Care Nephrology,You are admitting a 65-year-old kidney trans-p...,Which ONE of the following is the MOST importa...,A. Administration of a bolus of 30 ml/kg 0.9% ...,D,temp nan,Answer D: Early adminstration of appropriate ...,primary_eval
5,NEPHRO-EVAL-0006,2830,Acute Kidney Injury & Critical Care Nephrology,You are meeting with your hospital’s chief inf...,Which ONE of the following individuals is MOST...,A. A 35-year-old woman with serum creatinine t...,B,temp nan,Answer B: A 76-year-old woman with a serum cr...,primary_eval
6,NEPHRO-EVAL-0007,2831,Acute Kidney Injury & Critical Care Nephrology,The chief quality ofﬁcer at your institution a...,"Based on the existing literature, implementing...",A. A reduction in nephrotoxin exposure and eve...,B,temp nan,"Answer B: A reduction in RRT events, inpatien...",primary_eval
7,NEPHRO-EVAL-0008,2832,Acute Kidney Injury & Critical Care Nephrology,"Several weeks after joining a new practice, yo...",Which ONE of the following outcomes has been d...,"A. Fewer cases of severe AKI, but no differenc...",C,temp nan,"Answer C: Fewer cases of severe AKI, a higher...",primary_eval
8,NEPHRO-EVAL-0009,2833,Acute Kidney Injury & Critical Care Nephrology,NaN,Question 2833:\nImplementing a Kidney Disease ...,A. Shorter length of mechanical ventilation\nB...,B,temp nan,Answer B: A decreased rate of stage 2 and 3 AKI,primary_eval
9,NEPHRO-EVAL-0010,2834,Acute Kidney Injury & Critical Care Nephrology,A 28-year-old woman with ESRD receives a de-ce...,This patient’s urine output in response to int...,A. Acute renal allograft rejection\nB. 1-year ...,C,temp nan,Answer C: Delayed graft function after renal ...,primary_eval



HELD-OUT EVALUATION
Shape: (160, 9)
Columns: ['BenchmarkID', 'ID', 'Subject', 'Context', 'Question', 'Choices', 'Answer', 'Solution', 'explanation']


,BenchmarkID,ID,Subject,Context,Question,Choices,Answer,Solution,explanation
0,NEPHRO-HOLDOUT-0001,2828,Acute Kidney Injury & Critical Care Nephrology,A 63-year-old kidney transplant recipient is s...,"In this setting, administration of bicarbonate...",A. Increased risk of mortality\nB. Increased r...,C,temp nan,Answer C: Decreased risk of progressive AKI r...
1,NEPHRO-HOLDOUT-0002,2848,Acute Kidney Injury & Critical Care Nephrology,A 65-year-old man with stage G3a CKD is seen i...,Which ONE of the following should you tell his...,A. Off-pump surgery in eligible patients is as...,A,temp nan,Answer A: Off-pump surgery in eligible patien...
2,NEPHRO-HOLDOUT-0003,5444,Acute Kidney Injury & Critical Care Nephrology,\nA 72-year-old man with hepatitis C–related c...,Which ONE of the following is the MOST likely ...,1. Abdominal compartment syndrome\n2. Acute tu...,D,temp nan,D. Hepatorenal syndrome
3,NEPHRO-HOLDOUT-0004,5448,Acute Kidney Injury & Critical Care Nephrology,\nYou are asked to see a 63-year-old man in co...,Which ONE of the following is MOST likely to p...,1. A fractional excretion of urea > 50%\n2. A ...,C,temp nan,C. Urine volume <200 ml over 2 hours after 1....
4,NEPHRO-HOLDOUT-0005,5455,Acute Kidney Injury & Critical Care Nephrology,NaN,Question 5455\nIn which ONE of the following c...,1. Coronary artery bypass surgery\n2. Open abd...,C,temp nan,C. Radiocontrast exposure
5,NEPHRO-HOLDOUT-0006,5457,Acute Kidney Injury & Critical Care Nephrology,\nYou are asked to see a 68-year-old man with ...,Which ONE of the following is the MOST appropr...,1. Give a ﬂuid bolus and reassess in 1 hour\n2...,A,temp nan,A. Give a fluid bolus and reassess in 1 hour
6,NEPHRO-HOLDOUT-0007,5463,Acute Kidney Injury & Critical Care Nephrology,\nA 34-year-old combat victim is ﬂown to a mil...,Which ONE of the following places this trauma ...,1. Chest injury\n2. Head injury\n3. Abdominal ...,C,temp nan,C. Abdominal injury
7,NEPHRO-HOLDOUT-0008,5467,Acute Kidney Injury & Critical Care Nephrology,\nA 47-year-old woman with advanced heart fail...,Which ONE of the following would you advise re...,1. She is likely to experience at least a tran...,A,temp nan,A. She is likely to experience at least a tran...
8,NEPHRO-HOLDOUT-0009,5471,Acute Kidney Injury & Critical Care Nephrology,\nAn 80-year-old man is hospitalized for AKI. ...,Which ONE of the following diagnostic tests is...,1. Serum creatine kinase\n2. Urine eosinophils...,C,temp nan,C. Kidney ultrasound
9,NEPHRO-HOLDOUT-0010,5510,Acute Kidney Injury & Critical Care Nephrology,"\nA 75-year-old woman with diabetes, hypertens...",What strategies can you implement to prevent AKI?,A. Check urine sodium and risk-stratify...,C,temp nan,C. Give 0.9% sodium chloride before and after ...


# 4.4 Inspect benchmark structure and data quality

We now perform a more detailed structural check of the benchmark manifest.

The inspection covers:

- overall dataset shape,
- column names,
- sample records,
- data types, and
- missing values.

This confirms that the manifest contains the fields required for evaluation and split management, and helps identify any data-quality issues before the evaluation workflow continues.

The manifest contains **796 records across 10 columns**.

We specifically check for missing values rather than silently filling them, because the evaluation data should be preserved in its original form.

**Expected outcome:** the manifest structure and data types are consistent with the expected benchmark schema, with any missing values explicitly reported for review.

# 4.5 Validate official evaluation splits

### Confirming the Benchmark Split Assignments

Before proceeding with model evaluation and fine-tuning, we verify the official split assignments provided by the benchmark manifest.

The benchmark contains two evaluation splits:

- **`primary_eval` — 636 questions**
- **`held_out` — 160 questions**

There is no training split in the benchmark manifest.

We also inspect the subject distribution within each split to understand how the evaluation data is organized.

### Why this validation matters

The split assignments determine how the benchmark will be used throughout the experiment:

- `primary_eval` is used for the base-model baseline and the later base-vs-fine-tuned comparison.
- `held_out` is reserved for the separate generalization evaluation.
- Neither split is permitted to become part of the QLoRA training corpus.

This check establishes the data boundary before any fine-tuning begins.

**Expected outcome:** the manifest should contain exactly **636 primary-evaluation questions and 160 held-out questions**, for a total of **796 evaluation questions**.

In [29]:
# Compact manifest inspection

print("Manifest shape:", manifest_df.shape)
print("Manifest columns:", list(manifest_df.columns))

print("\nFirst 10 rows:")
display(manifest_df.head(10))

print("\nData types:")
display(manifest_df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(
    manifest_df.isna().sum()
    .sort_values(ascending=False)
    .to_frame("Missing")
)

Manifest shape: (796, 10)
Manifest columns: ['BenchmarkID', 'ID', 'Subject', 'Context', 'Question', 'Choices', 'Answer', 'Solution', 'explanation', 'split']

First 10 rows:


,BenchmarkID,ID,Subject,Context,Question,Choices,Answer,Solution,explanation,split
0,NEPHRO-EVAL-0001,2824,Acute Kidney Injury & Critical Care Nephrology,NaN,Question 2824:\nThe transfusion of longer-term...,1. Increased risk of transfusion-related react...,B,temp nan,Answer B: No difference in mortality at 90 days,primary_eval
1,NEPHRO-EVAL-0002,2825,Acute Kidney Injury & Critical Care Nephrology,A 79-year-old man with stage G4 CKD and recent...,"Based on data from the NUTRIREA-2 study, enter...",A. Decreased incidence of systemic bacteremia\...,B,temp nan,Answer B: Increased incidence of gastrointest...,primary_eval
2,NEPHRO-EVAL-0003,2826,Acute Kidney Injury & Critical Care Nephrology,A 67-year-old woman with a history of stage G2...,Which ONE of the following is CORRECT about th...,1. She is at decreased risk for inpatient mort...,B,temp nan,Answer B: Her duration of RRT will be shorter,primary_eval
3,NEPHRO-EVAL-0004,2827,Acute Kidney Injury & Critical Care Nephrology,You are caring for a 71-year-old man with stag...,Which ONE of the following is associated with ...,A. A decreased risk of surgical site infection...,B,temp nan,Answer B: An increased risk of AKI requiring RRT,primary_eval
4,NEPHRO-EVAL-0005,2829,Acute Kidney Injury & Critical Care Nephrology,You are admitting a 65-year-old kidney trans-p...,Which ONE of the following is the MOST importa...,A. Administration of a bolus of 30 ml/kg 0.9% ...,D,temp nan,Answer D: Early adminstration of appropriate ...,primary_eval
5,NEPHRO-EVAL-0006,2830,Acute Kidney Injury & Critical Care Nephrology,You are meeting with your hospital’s chief inf...,Which ONE of the following individuals is MOST...,A. A 35-year-old woman with serum creatinine t...,B,temp nan,Answer B: A 76-year-old woman with a serum cr...,primary_eval
6,NEPHRO-EVAL-0007,2831,Acute Kidney Injury & Critical Care Nephrology,The chief quality ofﬁcer at your institution a...,"Based on the existing literature, implementing...",A. A reduction in nephrotoxin exposure and eve...,B,temp nan,"Answer B: A reduction in RRT events, inpatien...",primary_eval
7,NEPHRO-EVAL-0008,2832,Acute Kidney Injury & Critical Care Nephrology,"Several weeks after joining a new practice, yo...",Which ONE of the following outcomes has been d...,"A. Fewer cases of severe AKI, but no differenc...",C,temp nan,"Answer C: Fewer cases of severe AKI, a higher...",primary_eval
8,NEPHRO-EVAL-0009,2833,Acute Kidney Injury & Critical Care Nephrology,NaN,Question 2833:\nImplementing a Kidney Disease ...,A. Shorter length of mechanical ventilation\nB...,B,temp nan,Answer B: A decreased rate of stage 2 and 3 AKI,primary_eval
9,NEPHRO-EVAL-0010,2834,Acute Kidney Injury & Critical Care Nephrology,A 28-year-old woman with ESRD receives a de-ce...,This patient’s urine output in response to int...,A. Acute renal allograft rejection\nB. 1-year ...,C,temp nan,Answer C: Delayed graft function after renal ...,primary_eval



Data types:


,dtype
BenchmarkID,object
ID,int64
Subject,object
Context,object
Question,object
Choices,object
Answer,object
Solution,object
explanation,object
split,object



Missing values:


,Missing
Context,48
BenchmarkID,0
ID,0
Subject,0
Question,0
Choices,0
Answer,0
Solution,0
explanation,0
split,0


In [30]:
# Inspect the official benchmark split assignments

print("Official split distribution")
print("===========================")

split_counts = (
    manifest_df["split"]
    .value_counts()
    .rename_axis("Split")
    .reset_index(name="Questions")
)

display(split_counts)

print("\nSubject distribution by split")
print("=============================")

split_subject_counts = (
    manifest_df
    .groupby(["split", "Subject"])
    .size()
    .reset_index(name="Questions")
)

display(split_subject_counts)

Official split distribution


,Split,Questions
0,primary_eval,636
1,held_out,160



Subject distribution by split


,split,Subject,Questions
0,held_out,Acute Kidney Injury & Critical Care Nephrology,18
1,held_out,Chronic Kidney Disease,6
2,held_out,Chronic Kidney Disease and Progression,6
3,held_out,Disorder of Divalent Ions Renal Bone Disease a...,6
4,held_out,Disorders of Divalent Ions Renal Bone Disease ...,5
5,held_out,"Disorders of Divalent Ions, Renal Bone Disease...",11
6,held_out,ESRD & Dialysis,6
7,held_out,Electrolyte Acid-Base Disorders,6
8,held_out,Electrolytes and Acid-Base Disorders,6
9,held_out,End Stage Renal Disease & Dialysis,6


# 7. Prepare fine-tuning dataset


# 7.1 Define training-data requirements

# 7.2 Load/create the separate nephrology training corpus

# 7.3 Check for evaluation-data leakage


# 7.4 Convert examples to instruction format


# 7.5 Inspect training examples

# 7.6 Create training/validation split

# 7.7 Report training dataset statistics

# 8. Fine-tune using QLoRA

# 8.1 Explain QLoRA configuration

# 8.2 Configure LoRA adapters


# 8.3 Configure training arguments


# 8.4 Train the model


# 8.5 Inspect training metrics

# 9. Save the LoRA adapter

# 9.1 Save adapter

# 9.2 Verify saved artifacts

# 9.3 Save tokenizer/configuration if required

# 10. Run the exact same evaluation on the fine-tuned model

# 10.1 Load the fine-tuned adapter

# 10.2 Reuse the same inference protocol

# 10.3 Reuse the same MCQ prompt

# 10.4 Reuse the same parser

# 10.5 Evaluate the same 636 questions


# 10.6 Aggregate results

# 11. Evaluate on held-out questions


# 11.1 Run inference on 160 held-out questions

# 11.2 Parse responses


# 11.3 Calculate held-out accuracy


# 11.4 Analyze unparsed responses

# 12. Compare base vs fine-tuned model


# 12.1 Overall accuracy comparison

# 12.2 Answered-only accuracy comparison


# 12.3 Unparsed-response comparison


# 12.4 Subject-level comparison


# 12.5 Error-pattern comparison


# 12.6 Held-out comparison

# 13. Document findings, limitations, and conclusions


# 13.1 Key findings


# 13.2 What improved


# 13.3 What did not improve

# 13.4 Limitations


# 13.5 Data leakage considerations


# 13.6 Conclusion 